In [ ]:
# ============================================================
# CELL 1 — INSTALL BASIC PITCH WITHOUT REINSTALLING NUMPY
# ============================================================

!pip install -q basic-pitch==0.4.0

In [ ]:
# ============================================================
# CELL 1 — INSTALL + IMPORTS
# ============================================================


import numpy as np
import pandas as pd
import librosa
from scipy.signal import find_peaks

from basic_pitch.inference import predict
from basic_pitch import ICASSP_2022_MODEL_PATH

import json
import base64

from IPython.display import HTML, display
from google.colab import files

In [ ]:
#Mounting a new folder from google colab onto drive
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# ============================================================
# CELL 2 — LOAD AUDIO + BASIC PITCH
# ============================================================

audio_path = "/content/drive/MyDrive/New Recording 18.m4a"

# Load audio for RMS analysis later
audio, sr = librosa.load(
    audio_path,
    sr=None,
    mono=True
)

# Run Basic Pitch
model_output, midi_data, note_events = predict(
    audio_path,
    ICASSP_2022_MODEL_PATH
)

print("Basic Pitch completed.")
print("Detected events:", len(note_events))

In [ ]:
# ============================================================
# CELL 3 — CLEAN BASIC PITCH EVENTS
# ============================================================

NOTE_NAMES = [
    "C", "C#", "D", "D#", "E", "F",
    "F#", "G", "G#", "A", "A#", "B"
]

def midi_to_note(midi):
    octave = (midi // 12) - 1
    note_name = NOTE_NAMES[midi % 12]
    return f"{note_name}{octave}"


# ------------------------------------------------------------
# Convert Basic Pitch output to DataFrame
# ------------------------------------------------------------

rows = []

for start, end, midi, amplitude, pitch_bends in note_events:

    rows.append({
        "Start": float(start),
        "End": float(end),
        "Duration": float(end - start),
        "MIDI": int(midi),
        "Amplitude": float(amplitude)
    })

bp_df = pd.DataFrame(rows)


# ------------------------------------------------------------
# Sort
# ------------------------------------------------------------

bp_df = bp_df.sort_values(
    "Start"
).reset_index(drop=True)


# ------------------------------------------------------------
# Convert MIDI → note name
# ------------------------------------------------------------

bp_df["Note"] = bp_df["MIDI"].apply(
    midi_to_note
)


# ------------------------------------------------------------
# Remove extremely short events
# ------------------------------------------------------------

MIN_DURATION = 0.15

bp_df = bp_df[
    bp_df["Duration"] >= MIN_DURATION
].copy()

bp_df = bp_df.reset_index(drop=True)


# ------------------------------------------------------------
# Merge consecutive events with same pitch
# ------------------------------------------------------------

MAX_GAP = 0.08

merged = []

for _, row in bp_df.iterrows():

    current = row.to_dict()

    if len(merged) == 0:
        merged.append(current)
        continue

    previous = merged[-1]

    gap = current["Start"] - previous["End"]

    if (
        current["MIDI"] == previous["MIDI"]
        and gap <= MAX_GAP
    ):

        previous["End"] = max(
            previous["End"],
            current["End"]
        )

        previous["Duration"] = (
            previous["End"] -
            previous["Start"]
        )

        previous["Amplitude"] = max(
            previous["Amplitude"],
            current["Amplitude"]
        )

    else:
        merged.append(current)


clean_df = pd.DataFrame(merged)

clean_df["Start"] = clean_df["Start"].round(3)
clean_df["End"] = clean_df["End"].round(3)
clean_df["Duration"] = clean_df["Duration"].round(3)
clean_df["Amplitude"] = clean_df["Amplitude"].round(3)

clean_df = clean_df[
    [
        "Note",
        "Start",
        "End",
        "Duration",
        "MIDI",
        "Amplitude"
    ]
]

print("Basic Pitch segments:", len(clean_df))

In [ ]:
# ============================================================
# CELL 4 — HYBRID BASIC PITCH + RMS SPLITTING
# ============================================================

# ------------------------------------------------------------
# SETTINGS
# ------------------------------------------------------------

PROMINENCE_THRESHOLD = 0.025

POSITION_MIN = 0.35
POSITION_MAX = 0.65

FRAME_LENGTH = 1024
HOP_LENGTH = 128

MIN_VALLEY_PROMINENCE = 0.001


# ------------------------------------------------------------
# RMS analysis
# ------------------------------------------------------------

def analyze_rms_segment(audio, sr, start, end):

    start_sample = int(start * sr)
    end_sample = int(end * sr)

    segment_audio = audio[
        start_sample:end_sample
    ]

    rms = librosa.feature.rms(
        y=segment_audio,
        frame_length=FRAME_LENGTH,
        hop_length=HOP_LENGTH
    )[0]

    if len(rms) < 3:
        return None

    rms_time = librosa.frames_to_time(
        np.arange(len(rms)),
        sr=sr,
        hop_length=HOP_LENGTH
    ) + start

    valleys, properties = find_peaks(
        -rms,
        prominence=MIN_VALLEY_PROMINENCE
    )

    if len(valleys) == 0:
        return None

    valley_times = rms_time[valleys]
    prominences = properties["prominences"]

    strongest_index = np.argmax(prominences)

    strongest_time = valley_times[
        strongest_index
    ]

    strongest_prominence = prominences[
        strongest_index
    ]

    valley_position = (
        (strongest_time - start) /
        (end - start)
    )

    return {
        "Strongest valley": strongest_time,
        "Strongest prominence": strongest_prominence,
        "Valley position": valley_position
    }


# ------------------------------------------------------------
# Determine whether to split
# ------------------------------------------------------------

def should_split(analysis):

    if analysis is None:
        return False

    prominence_pass = (
        analysis["Strongest prominence"]
        >= PROMINENCE_THRESHOLD
    )

    position_pass = (
        POSITION_MIN
        <= analysis["Valley position"]
        <= POSITION_MAX
    )

    return (
        prominence_pass
        and position_pass
    )


# ------------------------------------------------------------
# Process Basic Pitch segments
# ------------------------------------------------------------

final_segments = []

for _, row in clean_df.iterrows():

    note = row["Note"]
    start = float(row["Start"])
    end = float(row["End"])
    midi = int(row["MIDI"])
    amplitude = float(row["Amplitude"])

    analysis = analyze_rms_segment(
        audio,
        sr,
        start,
        end
    )

    split = should_split(analysis)


    # --------------------------------------------------------
    # Normal Basic Pitch segment
    # --------------------------------------------------------

    if not split:

        final_segments.append({

            "Note": note,
            "Start": start,
            "End": end,
            "Duration": end - start,
            "MIDI": midi,
            "Amplitude": amplitude,
            "Detection": "Basic Pitch",

            "RMS prominence":
                analysis["Strongest prominence"]
                if analysis is not None
                else np.nan,

            "Valley position":
                analysis["Valley position"]
                if analysis is not None
                else np.nan
        })


    # --------------------------------------------------------
    # Split into two repeated notes
    # --------------------------------------------------------

    else:

        split_time = analysis[
            "Strongest valley"
        ]

        # First note
        final_segments.append({

            "Note": note,
            "Start": start,
            "End": split_time,
            "Duration": split_time - start,
            "MIDI": midi,
            "Amplitude": amplitude,
            "Detection": "Basic Pitch + RMS split",
            "RMS prominence":
                analysis["Strongest prominence"],
            "Valley position":
                analysis["Valley position"]
        })

        # Second note
        final_segments.append({

            "Note": note,
            "Start": split_time,
            "End": end,
            "Duration": end - split_time,
            "MIDI": midi,
            "Amplitude": amplitude,
            "Detection": "Basic Pitch + RMS split",
            "RMS prominence":
                analysis["Strongest prominence"],
            "Valley position":
                analysis["Valley position"]
        })


# ------------------------------------------------------------
# Create final DataFrame
# ------------------------------------------------------------

final_df = pd.DataFrame(
    final_segments
)

final_df = final_df.sort_values(
    "Start"
).reset_index(drop=True)


print("Original Basic Pitch segments:", len(clean_df))
print("Final hybrid segments:", len(final_df))
print(
    "Segments split by RMS:",
    (
        final_df["Detection"]
        == "Basic Pitch + RMS split"
    ).sum()
)

In [ ]:
# ============================================================
# CELL 5 — CLEAN PROTOTYPE DATA
# ============================================================

# E5 was not played in this recording,
# so remove those known false detections.

prototype_df = final_df[
    final_df["Note"] != "E5"
].copy()

prototype_df = prototype_df.sort_values(
    "Start"
).reset_index(drop=True)


# ------------------------------------------------------------
# Round values
# ------------------------------------------------------------

prototype_df["Start"] = (
    prototype_df["Start"].round(3)
)

prototype_df["End"] = (
    prototype_df["End"].round(3)
)

prototype_df["Duration"] = (
    prototype_df["Duration"].round(3)
)


# ------------------------------------------------------------
# Save prototype data
# ------------------------------------------------------------

prototype_df.to_csv(
    "/content/melody_segments.csv",
    index=False
)


print("=" * 80)
print("PROTOTYPE MELODY DATA")
print("=" * 80)

print(
    prototype_df[
        [
            "Note",
            "Start",
            "End",
            "Duration",
            "MIDI",
            "Detection"
        ]
    ].to_string(index=False)
)

print("\nFinal prototype segments:",
      len(prototype_df))

print("\nSaved:")
print("/content/melody_segments.csv")

In [ ]:
# ============================================================
# CELL 6 — AUDIO-SYNCHRONIZED MELODY VISUALIZATION
# ============================================================

import json
import base64
from IPython.display import HTML, display


# ------------------------------------------------------------
# Prepare prototype data
# ------------------------------------------------------------

df = prototype_df.copy()

df = df.sort_values("Start").reset_index(drop=True)

notes = df[
    ["Note", "Start", "End", "MIDI"]
].to_dict("records")

notes_json = json.dumps(notes)


# ------------------------------------------------------------
# MIDI range
# ------------------------------------------------------------

min_midi = int(df["MIDI"].min()) - 1
max_midi = int(df["MIDI"].max()) + 1


# ------------------------------------------------------------
# Read audio
# ------------------------------------------------------------

with open(audio_path, "rb") as f:
    audio_bytes = f.read()

audio_base64 = base64.b64encode(
    audio_bytes
).decode("utf-8")


# ------------------------------------------------------------
# HTML
# ------------------------------------------------------------

html = """
<div style="
    width:100%;
    max-width:1000px;
    margin:auto;
    font-family:Arial,sans-serif;
">

    <audio
        id="melodyAudio"
        controls
        style="width:100%; margin-bottom:20px;"
    >
        <source
            src="data:audio/mp4;base64,AUDIO_DATA_HERE"
            type="audio/mp4"
        >
    </audio>


    <div style="
        position:relative;
        height:500px;
        border:1px solid #ccc;
        overflow:hidden;
        background:#fafafa;
    ">

        <div
            id="yAxis"
            style="
                position:absolute;
                left:0;
                top:0;
                width:60px;
                height:100%;
                border-right:1px solid #ccc;
                background:white;
                z-index:5;
            "
        ></div>


        <div
            id="melodyArea"
            style="
                position:absolute;
                left:60px;
                top:0;
                right:0;
                bottom:0;
            "
        >

            <div
                id="playhead"
                style="
                    position:absolute;
                    top:0;
                    bottom:0;
                    width:3px;
                    background:black;
                    z-index:10;
                    pointer-events:none;
                "
            ></div>

        </div>

    </div>

</div>


<script>

const notes = NOTES_DATA;

const minMidi = MIN_MIDI;
const maxMidi = MAX_MIDI;

const audio =
    document.getElementById("melodyAudio");

const melodyArea =
    document.getElementById("melodyArea");

const yAxis =
    document.getElementById("yAxis");

const playhead =
    document.getElementById("playhead");


// ============================================================
// TIME RANGE
// ============================================================

const startTime = Math.min(
    ...notes.map(n => n.Start)
);

const endTime = Math.max(
    ...notes.map(n => n.End)
);

const totalTime =
    endTime - startTime;


// ============================================================
// MIDI → NOTE NAME
// ============================================================

function midiToNote(midi) {

    const names = [
        "C", "C#", "D", "D#", "E", "F",
        "F#", "G", "G#", "A", "A#", "B"
    ];

    const octave =
        Math.floor(midi / 12) - 1;

    return names[midi % 12] + octave;
}


// ============================================================
// Y AXIS
// ============================================================

for (
    let midi = minMidi;
    midi <= maxMidi;
    midi++
) {

    const label =
        document.createElement("div");

    label.innerText =
        midiToNote(midi);

    label.style.position =
        "absolute";

    label.style.right =
        "8px";

    label.style.transform =
        "translateY(-50%)";

    label.style.fontSize =
        "13px";

    const y =
        (
            1 -
            (midi - minMidi) /
            (maxMidi - minMidi)
        ) * 100;

    label.style.top =
        y + "%";

    yAxis.appendChild(label);
}


// ============================================================
// MELODY BLOCKS
// ============================================================

notes.forEach((note, index) => {

    const block =
        document.createElement("div");

    block.className =
        "noteBlock";

    block.dataset.index =
        index;


    const left =
        (
            (note.Start - startTime) /
            totalTime
        ) * 100;

    const width =
        (
            (note.End - note.Start) /
            totalTime
        ) * 100;


    const top =
        (
            1 -
            (note.MIDI - minMidi) /
            (maxMidi - minMidi)
        ) * 100;


    block.style.position =
        "absolute";

    block.style.left =
        left + "%";

    block.style.width =
        width + "%";

    block.style.top =
        "calc(" + top + "% - 15px)";

    block.style.height =
        "30px";

    block.style.border =
        "1px solid #555";

    block.style.borderRadius =
        "5px";

    block.style.background =
        "#ddd";

    block.style.display =
        "flex";

    block.style.alignItems =
        "center";

    block.style.justifyContent =
        "center";

    block.style.fontSize =
        "12px";

    block.style.fontWeight =
        "bold";


    // Note name inside block

    block.innerText =
        note.Note;


    melodyArea.appendChild(
        block
    );

});


// ============================================================
// UPDATE VISUALIZATION FROM ACTUAL AUDIO POSITION
// ============================================================

function updateVisualization() {

    const currentTime =
        audio.currentTime;


    // --------------------------------------------------------
    // Move playhead
    // --------------------------------------------------------

    const position =
        (
            (currentTime - startTime) /
            totalTime
        ) * 100;

    playhead.style.left =
        position + "%";


    // --------------------------------------------------------
    // Highlight current note
    // --------------------------------------------------------

    notes.forEach(
        (note, index) => {

            const block =
                document.querySelector(
                    '.noteBlock[data-index="' +
                    index +
                    '"]'
                );


            if (
                currentTime >= note.Start &&
                currentTime <= note.End
            ) {

                block.style.background =
                    "#777";

                block.style.color =
                    "white";

                block.style.transform =
                    "scaleY(1.15)";

            } else {

                block.style.background =
                    "#ddd";

                block.style.color =
                    "black";

                block.style.transform =
                    "scaleY(1)";

            }

        }
    );


    // Keep animation synchronized to audio

    if (!audio.paused && !audio.ended) {

        requestAnimationFrame(
            updateVisualization
        );

    }

}


// ============================================================
// PLAY
// ============================================================

audio.addEventListener(
    "play",
    () => {

        requestAnimationFrame(
            updateVisualization
        );

    }
);


// ============================================================
// PAUSE
// ============================================================

audio.addEventListener(
    "pause",
    () => {

        updateVisualization();

    }
);


// ============================================================
// SEEK
// ============================================================

audio.addEventListener(
    "seeked",
    () => {

        updateVisualization();

    }
);

</script>
"""


# ------------------------------------------------------------
# Insert Python data into HTML
# ------------------------------------------------------------

html = html.replace(
    "NOTES_DATA",
    notes_json
)

html = html.replace(
    "MIN_MIDI",
    str(min_midi)
)

html = html.replace(
    "MAX_MIDI",
    str(max_midi)
)

html = html.replace(
    "AUDIO_DATA_HERE",
    audio_base64
)


# ------------------------------------------------------------
# Display
# ------------------------------------------------------------

display(HTML(html))

In [ ]:
import librosa
import numpy as np
import matplotlib.pyplot as plt

# ------------------------------------------------------------
# LOAD CLAP RECORDING
# ------------------------------------------------------------

clap_path = "/content/drive/MyDrive/Beat.m4a"

clap_audio, clap_sr = librosa.load(
    clap_path,
    sr=None,
    mono=True
)

print("Sample rate:", clap_sr)
print("Duration:", len(clap_audio) / clap_sr)

In [ ]:
import librosa
import numpy as np
import matplotlib.pyplot as plt
from scipy.signal import find_peaks

# ============================================================
# 1. LOAD CLAP RECORDING
# ============================================================

clap_path = "/content/drive/MyDrive/Beat.m4a"

y, sr = librosa.load(
    clap_path,
    sr=None,
    mono=True
)

print("Sample rate:", sr)
print("Duration:", round(len(y) / sr, 2), "seconds")


# ============================================================
# 2. CALCULATE ONSET STRENGTH
# ============================================================

onset_env = librosa.onset.onset_strength(
    y=y,
    sr=sr
)

times = librosa.times_like(
    onset_env,
    sr=sr
)


# ============================================================
# 3. DETECT STRONG PEAKS
#
# Your graph shows the actual claps as large peaks,
# generally above ~15.
# ============================================================

MIN_CLAP_STRENGTH = 15

# Minimum time between two claps
MIN_CLAP_INTERVAL = 0.30

# Convert seconds to onset frames
frame_interval = times[1] - times[0]

min_distance_frames = int(
    MIN_CLAP_INTERVAL / frame_interval
)

peaks, properties = find_peaks(
    onset_env,
    height=MIN_CLAP_STRENGTH,
    distance=min_distance_frames
)


# ============================================================
# 4. CONVERT PEAKS TO TIME
# ============================================================

clap_times = times[peaks]
clap_strengths = onset_env[peaks]


# ============================================================
# 5. PRINT RESULTS
# ============================================================

print("\nDETECTED CLAPS")
print("=" * 60)

for i, (t, strength) in enumerate(
    zip(clap_times, clap_strengths),
    start=1
):
    print(
        f"Clap {i:2d}: "
        f"{t:7.3f} sec   "
        f"strength = {strength:.2f}"
    )

print("\nNumber of detected claps:", len(clap_times))


# ============================================================
# 6. PLOT DETECTED CLAPS
# ============================================================

plt.figure(figsize=(16, 5))

plt.plot(
    times,
    onset_env,
    label="Onset strength"
)

plt.scatter(
    clap_times,
    clap_strengths,
    s=50,
    zorder=3,
    label="Detected claps"
)

plt.axhline(
    MIN_CLAP_STRENGTH,
    linestyle="--",
    label=f"Clap threshold = {MIN_CLAP_STRENGTH}"
)

plt.xlabel("Time (seconds)")
plt.ylabel("Onset strength")
plt.title("Detected Clap Times")

plt.legend()

plt.show()

In [ ]:
# ============================================================
# MELODY + BEAT REAL-TIME VISUALIZATION
# ============================================================

from IPython.display import HTML, display
import base64
import json
import pandas as pd
import numpy as np

# ------------------------------------------------------------
# 1. AUDIO FILE
# ------------------------------------------------------------

audio_path = "/content/drive/MyDrive/New Recording 18.m4a"


# ------------------------------------------------------------
# 2. PREPARE MELODY DATA
# ------------------------------------------------------------

melody_data = []

for _, row in prototype_df.iterrows():

    melody_data.append({
        "note": str(row["Note"]),
        "start": float(row["Start"]),
        "end": float(row["End"]),
        "midi": int(row["MIDI"])
    })


# ------------------------------------------------------------
# 3. PREPARE BEAT DATA
# ------------------------------------------------------------

beat_data = [
    float(t)
    for t in clap_times
]


# ------------------------------------------------------------
# 4. CONVERT AUDIO TO BASE64
#
# This lets the browser play the Colab audio directly.
# ------------------------------------------------------------

with open(audio_path, "rb") as f:
    audio_bytes = f.read()

audio_base64 = base64.b64encode(
    audio_bytes
).decode("utf-8")


# ------------------------------------------------------------
# 5. FIND NOTE RANGE
# ------------------------------------------------------------

midi_values = [
    x["midi"]
    for x in melody_data
]

min_midi = min(midi_values) - 1
max_midi = max(midi_values) + 1

note_names = {
    60: "C4",
    61: "C#4",
    62: "D4",
    63: "D#4",
    64: "E4",
    65: "F4",
    66: "F#4",
    67: "G4",
    68: "G#4",
    69: "A4",
    70: "A#4",
    71: "B4",
    72: "C5",
    73: "C#5",
    74: "D5",
    75: "D#5",
    76: "E5"
}

display_notes = []

for midi in range(max_midi, min_midi - 1, -1):

    if midi in note_names:
        display_notes.append(
            (midi, note_names[midi])
        )


# ------------------------------------------------------------
# 6. HTML + JAVASCRIPT VISUALIZATION
# ------------------------------------------------------------

html = f"""
<div style="
    width:100%;
    max-width:1100px;
    margin:auto;
    font-family:Arial,sans-serif;
">

    <audio id="melodyAudio" controls style="width:100%;">
        <source src="data:audio/wav;base64,{audio_base64}" type="audio/wav">
    </audio>

    <canvas
        id="melodyCanvas"
        width="1100"
        height="650"
        style="
            width:100%;
            border:1px solid #ccc;
            margin-top:15px;
        ">
    </canvas>

</div>

<script>

const canvas = document.getElementById("melodyCanvas");
const ctx = canvas.getContext("2d");

const audio = document.getElementById("melodyAudio");

const melody = {json.dumps(melody_data)};
const beats = {json.dumps(beat_data)};

const minMidi = {min_midi};
const maxMidi = {max_midi};

const noteNames = {json.dumps(note_names)};


// ============================================================
// VISUALIZATION SETTINGS
// ============================================================

const leftMargin = 80;
const rightMargin = 20;
const topMargin = 30;
const bottomMargin = 90;

const melodyHeight =
    canvas.height - topMargin - bottomMargin;

const beatY =
    canvas.height - 45;


// ============================================================
// FIND TOTAL TIME
// ============================================================

let totalTime = 0;

melody.forEach(note => {{
    totalTime = Math.max(totalTime, note.end);
}});

beats.forEach(beat => {{
    totalTime = Math.max(totalTime, beat);
}});


// ============================================================
// MIDI → Y POSITION
// ============================================================

function midiToY(midi) {{

    const range = maxMidi - minMidi;

    return topMargin +
        (maxMidi - midi) /
        range *
        melodyHeight;
}}


// ============================================================
// TIME → X POSITION
// ============================================================

function timeToX(time) {{

    const width =
        canvas.width -
        leftMargin -
        rightMargin;

    return leftMargin +
        (time / totalTime) * width;
}}


// ============================================================
// DRAW VISUALIZATION
// ============================================================

function draw() {{

    const currentTime = audio.currentTime;

    // --------------------------------------------------------
    // BACKGROUND
    // --------------------------------------------------------

    ctx.clearRect(
        0,
        0,
        canvas.width,
        canvas.height
    );

    ctx.fillStyle = "#fff";
    ctx.fillRect(
        0,
        0,
        canvas.width,
        canvas.height
    );


    // --------------------------------------------------------
    // TITLE
    // --------------------------------------------------------

    ctx.fillStyle = "#222";
    ctx.font = "bold 20px Arial";

    ctx.fillText(
        "Melody + Beat Visualization",
        leftMargin,
        20
    );


    // --------------------------------------------------------
    // NOTE LANES
    // --------------------------------------------------------

    ctx.font = "14px Arial";

    for (
        let midi = maxMidi;
        midi >= minMidi;
        midi--
    ) {{

        const y = midiToY(midi);

        // lane
        ctx.fillStyle =
            midi % 2 === 0
            ? "#fff5f5"
            : "#f7e5e5";

        ctx.fillRect(
            leftMargin,
            y - 18,
            canvas.width -
                leftMargin -
                rightMargin,
            36
        );

        // note name
        if (noteNames[midi]) {{

            ctx.fillStyle = "#333";

            ctx.fillText(
                noteNames[midi],
                15,
                y + 5
            );
        }}
    }}


    // --------------------------------------------------------
    // MELODY BLOCKS
    // --------------------------------------------------------

    melody.forEach(note => {{

        const x1 = timeToX(note.start);
        const x2 = timeToX(note.end);

        const y = midiToY(note.midi);

        const isPlaying =
            currentTime >= note.start &&
            currentTime <= note.end;

        // block
        ctx.fillStyle =
            isPlaying
            ? "#c94f7c"
            : "#d987a5";

        ctx.fillRect(
            x1,
            y - 14,
            Math.max(x2 - x1, 3),
            28
        );

        // outline
        ctx.strokeStyle = "#9d506f";

        ctx.strokeRect(
            x1,
            y - 14,
            Math.max(x2 - x1, 3),
            28
        );

        // note label
        ctx.fillStyle = "#222";
        ctx.font = "bold 12px Arial";

        if (x2 - x1 > 30) {{

            ctx.fillText(
                note.note,
                x1 + 5,
                y + 4
            );
        }}
    }});


    // --------------------------------------------------------
    // BEAT AREA
    // --------------------------------------------------------

    ctx.fillStyle = "#333";
    ctx.font = "bold 15px Arial";

    ctx.fillText(
        "BEAT",
        15,
        beatY + 5
    );


    // beat baseline
    ctx.strokeStyle = "#999";
    ctx.lineWidth = 1;

    ctx.beginPath();

    ctx.moveTo(
        leftMargin,
        beatY
    );

    ctx.lineTo(
        canvas.width - rightMargin,
        beatY
    );

    ctx.stroke();


    // --------------------------------------------------------
    // BEAT MARKERS
    // --------------------------------------------------------

    beats.forEach((beat, index) => {{

        const x = timeToX(beat);

        const distance =
            Math.abs(currentTime - beat);

        // highlight beat when audio is near it
        const active =
            distance < 0.12;

        ctx.beginPath();

        ctx.arc(
            x,
            beatY,
            active ? 10 : 6,
            0,
            Math.PI * 2
        );

        ctx.fillStyle =
            active
            ? "#c94f7c"
            : "#777";

        ctx.fill();

        // beat number
        ctx.fillStyle = "#555";
        ctx.font = "10px Arial";

        ctx.fillText(
            index + 1,
            x - 3,
            beatY + 25
        );
    }});


    // --------------------------------------------------------
    // CURRENT PLAYBACK POSITION
    // --------------------------------------------------------

    const playheadX =
        timeToX(currentTime);

    ctx.strokeStyle = "#222";
    ctx.lineWidth = 2;

    ctx.beginPath();

    ctx.moveTo(
        playheadX,
        topMargin
    );

    ctx.lineTo(
        playheadX,
        beatY + 15
    );

    ctx.stroke();


    // --------------------------------------------------------
    // CURRENT TIME
    // --------------------------------------------------------

    ctx.fillStyle = "#222";
    ctx.font = "14px Arial";

    ctx.fillText(
        currentTime.toFixed(2) +
        " / " +
        totalTime.toFixed(2) +
        " sec",
        canvas.width - 130,
        20
    );


    // --------------------------------------------------------
    // IMPORTANT:
    // Animation is driven by AUDIO PLAYBACK POSITION.
    // There is NO independent animation timer.
    // --------------------------------------------------------

    requestAnimationFrame(draw);
}}


// Start visualization

draw();

</script>
"""

display(HTML(html))

beat Offsett by .1 seconds in next code and it aligns much better to audio

In [ ]:
# ============================================================
# MELODY + BEAT REAL-TIME VISUALIZATION
# ============================================================

from IPython.display import HTML, display
import base64
import json
import pandas as pd
import numpy as np


# ============================================================
# 1. AUDIO FILE
# ============================================================

audio_path = "/content/drive/MyDrive/New Recording 18.m4a"


# ============================================================
# 2. BEAT TIMING CORRECTION
# ============================================================
#
# Negative = move beat earlier
# Positive = move beat later
#
# Start with -0.10 seconds.
# ============================================================

BEAT_OFFSET = -0.10


# ============================================================
# 3. PREPARE MELODY DATA
# ============================================================

melody_data = []

for _, row in prototype_df.iterrows():

    melody_data.append({
        "note": str(row["Note"]),
        "start": float(row["Start"]),
        "end": float(row["End"]),
        "midi": int(row["MIDI"])
    })


# ============================================================
# 4. PREPARE BEAT DATA
# ============================================================

beat_data = [
    max(0, float(t) + BEAT_OFFSET)
    for t in clap_times
]


# ============================================================
# 5. FIND NOTE RANGE
# ============================================================

midi_values = [
    x["midi"]
    for x in melody_data
]

min_midi = min(midi_values) - 1
max_midi = max(midi_values) + 1


# ============================================================
# 6. NOTE NAMES
# ============================================================

note_names = {
    60: "C4",
    61: "C#4",
    62: "D4",
    63: "D#4",
    64: "E4",
    65: "F4",
    66: "F#4",
    67: "G4",
    68: "G#4",
    69: "A4",
    70: "A#4",
    71: "B4",
    72: "C5",
    73: "C#5",
    74: "D5",
    75: "D#5",
    76: "E5"
}


# ============================================================
# 7. CONVERT AUDIO TO BASE64
# ============================================================

with open(audio_path, "rb") as f:
    audio_bytes = f.read()

audio_base64 = base64.b64encode(
    audio_bytes
).decode("utf-8")


# ============================================================
# 8. CREATE VISUALIZATION
# ============================================================

html = f"""
<div style="
    width:100%;
    max-width:1100px;
    margin:auto;
    font-family:Arial,sans-serif;
">

    <audio id="melodyAudio" controls style="width:100%;">
        <source
            src="data:audio/wav;base64,{audio_base64}"
            type="audio/wav">
    </audio>

    <canvas
        id="melodyCanvas"
        width="1100"
        height="650"
        style="
            width:100%;
            border:1px solid #ccc;
            margin-top:15px;
        ">
    </canvas>

</div>

<script>

const canvas =
    document.getElementById("melodyCanvas");

const ctx =
    canvas.getContext("2d");

const audio =
    document.getElementById("melodyAudio");


const melody =
    {json.dumps(melody_data)};

const beats =
    {json.dumps(beat_data)};


const minMidi = {min_midi};
const maxMidi = {max_midi};


const noteNames =
    {json.dumps(note_names)};


// ============================================================
// VISUALIZATION SETTINGS
// ============================================================

const leftMargin = 80;
const rightMargin = 20;
const topMargin = 40;
const bottomMargin = 100;

const melodyHeight =
    canvas.height -
    topMargin -
    bottomMargin;

const beatY =
    canvas.height - 55;


// ============================================================
// TOTAL TIME
// ============================================================

let totalTime = 0;

melody.forEach(note => {{
    totalTime =
        Math.max(totalTime, note.end);
}});

beats.forEach(beat => {{
    totalTime =
        Math.max(totalTime, beat);
}});


// ============================================================
// MIDI → Y
// ============================================================

function midiToY(midi) {{

    const range =
        maxMidi - minMidi;

    return topMargin +
        ((maxMidi - midi) / range) *
        melodyHeight;
}}


// ============================================================
// TIME → X
// ============================================================

function timeToX(time) {{

    const width =
        canvas.width -
        leftMargin -
        rightMargin;

    return leftMargin +
        (time / totalTime) * width;
}}


// ============================================================
// DRAW
// ============================================================

function draw() {{

    const currentTime =
        audio.currentTime;


    // --------------------------------------------------------
    // BACKGROUND
    // --------------------------------------------------------

    ctx.clearRect(
        0,
        0,
        canvas.width,
        canvas.height
    );

    ctx.fillStyle = "#ffffff";

    ctx.fillRect(
        0,
        0,
        canvas.width,
        canvas.height
    );


    // --------------------------------------------------------
    // TITLE
    // --------------------------------------------------------

    ctx.fillStyle = "#222";
    ctx.font = "bold 20px Arial";

    ctx.fillText(
        "Melody + Beat Visualization",
        leftMargin,
        25
    );


    // --------------------------------------------------------
    // NOTE LANES
    // --------------------------------------------------------

    for (
        let midi = maxMidi;
        midi >= minMidi;
        midi--
    ) {{

        const y =
            midiToY(midi);


        ctx.fillStyle =
            midi % 2 === 0
            ? "#fff5f5"
            : "#f7e5e5";


        ctx.fillRect(
            leftMargin,
            y - 18,
            canvas.width -
                leftMargin -
                rightMargin,
            36
        );


        // note name
        if (noteNames[midi]) {{

            ctx.fillStyle = "#333";
            ctx.font = "14px Arial";

            ctx.fillText(
                noteNames[midi],
                15,
                y + 5
            );
        }}
    }}


    // --------------------------------------------------------
    // MELODY BLOCKS
    // --------------------------------------------------------

    melody.forEach(note => {{

        const x1 =
            timeToX(note.start);

        const x2 =
            timeToX(note.end);

        const y =
            midiToY(note.midi);


        const isPlaying =
            currentTime >= note.start &&
            currentTime <= note.end;


        // note block
        ctx.fillStyle =
            isPlaying
            ? "#c94f7c"
            : "#d987a5";


        ctx.fillRect(
            x1,
            y - 14,
            Math.max(x2 - x1, 3),
            28
        );


        // outline
        ctx.strokeStyle =
            "#9d506f";

        ctx.strokeRect(
            x1,
            y - 14,
            Math.max(x2 - x1, 3),
            28
        );


        // note name inside block
        if (x2 - x1 > 30) {{

            ctx.fillStyle = "#222";
            ctx.font = "bold 12px Arial";

            ctx.fillText(
                note.note,
                x1 + 5,
                y + 4
            );
        }}
    }});


    // ========================================================
    // BEAT SECTION
    // ========================================================

    ctx.fillStyle = "#333";
    ctx.font = "bold 15px Arial";

    ctx.fillText(
        "BEAT",
        15,
        beatY + 5
    );


    // beat baseline
    ctx.strokeStyle = "#aaa";
    ctx.lineWidth = 1;

    ctx.beginPath();

    ctx.moveTo(
        leftMargin,
        beatY
    );

    ctx.lineTo(
        canvas.width - rightMargin,
        beatY
    );

    ctx.stroke();


    // --------------------------------------------------------
    // BEAT MARKERS
    // --------------------------------------------------------

    beats.forEach((beat, index) => {{

        const x =
            timeToX(beat);


        const distance =
            Math.abs(currentTime - beat);


        // Beat becomes active when audio reaches it
        const active =
            distance < 0.12;


        ctx.beginPath();

        ctx.arc(
            x,
            beatY,
            active ? 11 : 6,
            0,
            Math.PI * 2
        );


        ctx.fillStyle =
            active
            ? "#c94f7c"
            : "#777";


        ctx.fill();


        // beat number
        ctx.fillStyle = "#555";
        ctx.font = "10px Arial";

        ctx.fillText(
            index + 1,
            x - 3,
            beatY + 22
        );
    }});


    // ========================================================
    // PLAYHEAD
    // ========================================================

    const playheadX =
        timeToX(currentTime);


    ctx.strokeStyle = "#222";
    ctx.lineWidth = 2;

    ctx.beginPath();

    ctx.moveTo(
        playheadX,
        topMargin
    );

    ctx.lineTo(
        playheadX,
        beatY + 15
    );

    ctx.stroke();


    // ========================================================
    // CURRENT TIME
    // ========================================================

    ctx.fillStyle = "#222";
    ctx.font = "14px Arial";

    ctx.fillText(
        currentTime.toFixed(2) +
        " / " +
        totalTime.toFixed(2) +
        " sec",
        canvas.width - 140,
        25
    );


    // ========================================================
    // NEXT FRAME
    // ========================================================

    requestAnimationFrame(draw);
}}


// ============================================================
// START
// ============================================================

draw();

</script>
"""

display(HTML(html))

The one above I like visually the most, the one below has larger pulse, and the one below that is smaller pulse

In [ ]:
from IPython.display import HTML, display
import base64
import json
import pandas as pd
import numpy as np


# ============================================================
# 1. AUDIO FILE
# ============================================================

audio_path = "/content/drive/MyDrive/New Recording 18.m4a"


# ============================================================
# 2. BEAT TIMING OFFSET
# ============================================================

BEAT_OFFSET = -0.25


# ============================================================
# 3. PREPARE MELODY DATA
# ============================================================

melody_data = []

for _, row in prototype_df.iterrows():

    melody_data.append({
        "note": str(row["Note"]),
        "start": float(row["Start"]),
        "end": float(row["End"]),
        "midi": int(row["MIDI"])
    })


# ============================================================
# 4. PREPARE BEAT DATA
# ============================================================

beat_data = [
    max(0, float(t) + BEAT_OFFSET)
    for t in clap_times
]


# ============================================================
# 5. MIDI RANGE
# ============================================================

midi_values = [x["midi"] for x in melody_data]

min_midi = min(midi_values) - 1
max_midi = max(midi_values) + 1


# ============================================================
# 6. NOTE NAMES
# ============================================================

note_names = {
    60: "C4",
    61: "C#4",
    62: "D4",
    63: "D#4",
    64: "E4",
    65: "F4",
    66: "F#4",
    67: "G4",
    68: "G#4",
    69: "A4",
    70: "A#4",
    71: "B4",
    72: "C5",
    73: "C#5",
    74: "D5",
    75: "D#5",
    76: "E5"
}


# ============================================================
# 7. AUDIO → BASE64
# ============================================================

with open(audio_path, "rb") as f:
    audio_bytes = f.read()

audio_base64 = base64.b64encode(audio_bytes).decode("utf-8")


# ============================================================
# 8. HTML + JAVASCRIPT
# ============================================================

html = f"""
<div style="
    width:100%;
    max-width:1100px;
    margin:auto;
    font-family:Arial,sans-serif;
">

    <audio id="melodyAudio" controls style="width:100%;">
        <source
            src="data:audio/wav;base64,{audio_base64}"
            type="audio/wav">
    </audio>

    <canvas
        id="melodyCanvas"
        width="1100"
        height="650"
        style="
            width:100%;
            border:1px solid #ccc;
            margin-top:15px;
        ">
    </canvas>

</div>

<script>

const canvas = document.getElementById("melodyCanvas");
const ctx = canvas.getContext("2d");

const audio = document.getElementById("melodyAudio");

const melody = {json.dumps(melody_data)};
const beats = {json.dumps(beat_data)};

const minMidi = {min_midi};
const maxMidi = {max_midi};

const noteNames = {json.dumps(note_names)};


// ============================================================
// SETTINGS
// ============================================================

const leftMargin = 80;
const rightMargin = 20;
const topMargin = 40;
const bottomMargin = 100;

const melodyHeight =
    canvas.height - topMargin - bottomMargin;

const beatY = canvas.height - 55;


// ============================================================
// TOTAL TIME
// ============================================================

let totalTime = 0;

melody.forEach(note => {{
    totalTime = Math.max(totalTime, note.end);
}});

beats.forEach(beat => {{
    totalTime = Math.max(totalTime, beat);
}});


// ============================================================
// MIDI → Y
// ============================================================

function midiToY(midi) {{

    const range = maxMidi - minMidi;

    return topMargin +
        ((maxMidi - midi) / range) * melodyHeight;
}}


// ============================================================
// TIME → X
// ============================================================

function timeToX(time) {{

    const width =
        canvas.width - leftMargin - rightMargin;

    return leftMargin +
        (time / totalTime) * width;
}}


// ============================================================
// DRAW
// ============================================================

function draw() {{

    const currentTime = audio.currentTime;


    // --------------------------------------------------------
    // BACKGROUND
    // --------------------------------------------------------

    ctx.clearRect(
        0,
        0,
        canvas.width,
        canvas.height
    );

    ctx.fillStyle = "#ffffff";

    ctx.fillRect(
        0,
        0,
        canvas.width,
        canvas.height
    );


    // --------------------------------------------------------
    // TITLE
    // --------------------------------------------------------

    ctx.fillStyle = "#222";
    ctx.font = "bold 20px Arial";

    ctx.fillText(
        "Melody + Beat Visualization",
        leftMargin,
        25
    );


    // --------------------------------------------------------
    // NOTE LANES
    // --------------------------------------------------------

    for (
        let midi = maxMidi;
        midi >= minMidi;
        midi--
    ) {{

        const y = midiToY(midi);

        ctx.fillStyle =
            midi % 2 === 0
            ? "#fff5f5"
            : "#f7e5e5";

        ctx.fillRect(
            leftMargin,
            y - 18,
            canvas.width - leftMargin - rightMargin,
            36
        );


        if (noteNames[midi]) {{

            ctx.fillStyle = "#333";
            ctx.font = "14px Arial";

            ctx.fillText(
                noteNames[midi],
                15,
                y + 5
            );
        }}
    }}


    // --------------------------------------------------------
    // MELODY BLOCKS
    // --------------------------------------------------------

    melody.forEach(note => {{

        const x1 = timeToX(note.start);
        const x2 = timeToX(note.end);

        const y = midiToY(note.midi);

        const isPlaying =
            currentTime >= note.start &&
            currentTime <= note.end;


        ctx.fillStyle =
            isPlaying
            ? "#c94f7c"
            : "#d987a5";

        ctx.fillRect(
            x1,
            y - 14,
            Math.max(x2 - x1, 3),
            28
        );


        ctx.strokeStyle = "#9d506f";

        ctx.strokeRect(
            x1,
            y - 14,
            Math.max(x2 - x1, 3),
            28
        );


        if (x2 - x1 > 30) {{

            ctx.fillStyle = "#222";
            ctx.font = "bold 12px Arial";

            ctx.fillText(
                note.note,
                x1 + 5,
                y + 4
            );
        }}
    }});


    // ========================================================
    // BEAT SECTION
    // ========================================================

    ctx.fillStyle = "#333";
    ctx.font = "bold 15px Arial";

    ctx.fillText(
        "BEAT",
        15,
        beatY + 5
    );


    // Beat baseline

    ctx.strokeStyle = "#aaa";
    ctx.lineWidth = 1;

    ctx.beginPath();

    ctx.moveTo(
        leftMargin,
        beatY
    );

    ctx.lineTo(
        canvas.width - rightMargin,
        beatY
    );

    ctx.stroke();


    // ========================================================
    // PULSING BEATS
    // ========================================================

    beats.forEach((beat, index) => {{

        const x = timeToX(beat);

        const distance = currentTime - beat;


        // Active shortly AFTER the beat occurs

        const active =
            distance >= 0 &&
            distance < 0.25;


        let radius = 6;


        if (active) {{

            const progress =
                distance / 0.25;

            radius =
                13 - (progress * 7);
        }}


        // ----------------------------------------------------
        // Circle
        // ----------------------------------------------------

        ctx.beginPath();

        ctx.arc(
            x,
            beatY,
            radius,
            0,
            Math.PI * 2
        );


        ctx.fillStyle =
            active
            ? "#c94f7c"
            : "#777";

        ctx.fill();


        // ----------------------------------------------------
        // Beat number
        // ----------------------------------------------------

        ctx.fillStyle =
            active
            ? "#c94f7c"
            : "#555";

        ctx.font =
            active
            ? "bold 12px Arial"
            : "10px Arial";

        ctx.fillText(
            index + 1,
            x - 3,
            beatY + 22
        );

    }});


    // ========================================================
    // PLAYHEAD
    // ========================================================

    const playheadX =
        timeToX(currentTime);

    ctx.strokeStyle = "#222";
    ctx.lineWidth = 2;

    ctx.beginPath();

    ctx.moveTo(
        playheadX,
        topMargin
    );

    ctx.lineTo(
        playheadX,
        beatY + 15
    );

    ctx.stroke();


    // ========================================================
    // CURRENT TIME
    // ========================================================

    ctx.fillStyle = "#222";
    ctx.font = "14px Arial";

    ctx.fillText(
        currentTime.toFixed(2) +
        " / " +
        totalTime.toFixed(2) +
        " sec",
        canvas.width - 140,
        25
    );


    // ========================================================
    // CONTINUE ANIMATION
    // ========================================================

    requestAnimationFrame(draw);

}}


// ============================================================
// START DRAWING
// ============================================================

draw();

</script>
"""

display(HTML(html))

In [ ]:
from IPython.display import HTML, display
import base64
import json
import pandas as pd
import numpy as np


# ============================================================
# 1. AUDIO FILE
# ============================================================

audio_path = "/content/drive/MyDrive/New Recording 18.m4a"


# ============================================================
# 2. BEAT TIMING OFFSET
# ============================================================

BEAT_OFFSET = -0.25


# ============================================================
# 3. PREPARE MELODY DATA
# ============================================================

melody_data = []

for _, row in prototype_df.iterrows():

    melody_data.append({
        "note": str(row["Note"]),
        "start": float(row["Start"]),
        "end": float(row["End"]),
        "midi": int(row["MIDI"])
    })


# ============================================================
# 4. PREPARE BEAT DATA
# ============================================================

beat_data = [
    max(0, float(t) + BEAT_OFFSET)
    for t in clap_times
]


# ============================================================
# 5. MIDI RANGE
# ============================================================

midi_values = [x["midi"] for x in melody_data]

min_midi = min(midi_values) - 1
max_midi = max(midi_values) + 1


# ============================================================
# 6. NOTE NAMES
# ============================================================

note_names = {
    60: "C4",
    61: "C#4",
    62: "D4",
    63: "D#4",
    64: "E4",
    65: "F4",
    66: "F#4",
    67: "G4",
    68: "G#4",
    69: "A4",
    70: "A#4",
    71: "B4",
    72: "C5",
    73: "C#5",
    74: "D5",
    75: "D#5",
    76: "E5"
}


# ============================================================
# 7. AUDIO → BASE64
# ============================================================

with open(audio_path, "rb") as f:
    audio_bytes = f.read()

audio_base64 = base64.b64encode(audio_bytes).decode("utf-8")


# ============================================================
# 8. HTML + JAVASCRIPT
# ============================================================

html = f"""
<div style="
    width:100%;
    max-width:1100px;
    margin:auto;
    font-family:Arial,sans-serif;
">

    <audio id="melodyAudio" controls style="width:100%;">
        <source
            src="data:audio/wav;base64,{audio_base64}"
            type="audio/wav">
    </audio>

    <canvas
        id="melodyCanvas"
        width="1100"
        height="650"
        style="
            width:100%;
            border:1px solid #ccc;
            margin-top:15px;
        ">
    </canvas>

</div>

<script>

const canvas = document.getElementById("melodyCanvas");
const ctx = canvas.getContext("2d");

const audio = document.getElementById("melodyAudio");

const melody = {json.dumps(melody_data)};
const beats = {json.dumps(beat_data)};

const minMidi = {min_midi};
const maxMidi = {max_midi};

const noteNames = {json.dumps(note_names)};


// ============================================================
// SETTINGS
// ============================================================

const leftMargin = 80;
const rightMargin = 20;
const topMargin = 40;
const bottomMargin = 100;

const melodyHeight =
    canvas.height - topMargin - bottomMargin;

const beatY = canvas.height - 55;


// ============================================================
// TOTAL TIME
// ============================================================

let totalTime = 0;

melody.forEach(note => {{
    totalTime = Math.max(totalTime, note.end);
}});

beats.forEach(beat => {{
    totalTime = Math.max(totalTime, beat);
}});


// ============================================================
// MIDI → Y
// ============================================================

function midiToY(midi) {{

    const range = maxMidi - minMidi;

    return topMargin +
        ((maxMidi - midi) / range) * melodyHeight;
}}


// ============================================================
// TIME → X
// ============================================================

function timeToX(time) {{

    const width =
        canvas.width - leftMargin - rightMargin;

    return leftMargin +
        (time / totalTime) * width;
}}


// ============================================================
// DRAW
// ============================================================

function draw() {{

    const currentTime = audio.currentTime;


    // --------------------------------------------------------
    // BACKGROUND
    // --------------------------------------------------------

    ctx.clearRect(
        0,
        0,
        canvas.width,
        canvas.height
    );

    ctx.fillStyle = "#ffffff";

    ctx.fillRect(
        0,
        0,
        canvas.width,
        canvas.height
    );


    // --------------------------------------------------------
    // TITLE
    // --------------------------------------------------------

    ctx.fillStyle = "#222";
    ctx.font = "bold 20px Arial";

    ctx.fillText(
        "Melody + Beat Visualization",
        leftMargin,
        25
    );


    // --------------------------------------------------------
    // NOTE LANES
    // --------------------------------------------------------

    for (
        let midi = maxMidi;
        midi >= minMidi;
        midi--
    ) {{

        const y = midiToY(midi);

        ctx.fillStyle =
            midi % 2 === 0
            ? "#fff5f5"
            : "#f7e5e5";

        ctx.fillRect(
            leftMargin,
            y - 18,
            canvas.width - leftMargin - rightMargin,
            36
        );


        if (noteNames[midi]) {{

            ctx.fillStyle = "#333";
            ctx.font = "14px Arial";

            ctx.fillText(
                noteNames[midi],
                15,
                y + 5
            );
        }}
    }}


    // --------------------------------------------------------
    // MELODY BLOCKS
    // --------------------------------------------------------

    melody.forEach(note => {{

        const x1 = timeToX(note.start);
        const x2 = timeToX(note.end);

        const y = midiToY(note.midi);

        const isPlaying =
            currentTime >= note.start &&
            currentTime <= note.end;


        ctx.fillStyle =
            isPlaying
            ? "#c94f7c"
            : "#d987a5";

        ctx.fillRect(
            x1,
            y - 14,
            Math.max(x2 - x1, 3),
            28
        );


        ctx.strokeStyle = "#9d506f";

        ctx.strokeRect(
            x1,
            y - 14,
            Math.max(x2 - x1, 3),
            28
        );


        if (x2 - x1 > 30) {{

            ctx.fillStyle = "#222";
            ctx.font = "bold 12px Arial";

            ctx.fillText(
                note.note,
                x1 + 5,
                y + 4
            );
        }}
    }});


    // ========================================================
    // BEAT SECTION
    // ========================================================

    ctx.fillStyle = "#333";
    ctx.font = "bold 15px Arial";

    ctx.fillText(
        "BEAT",
        15,
        beatY + 5
    );


    // Beat baseline

    ctx.strokeStyle = "#aaa";
    ctx.lineWidth = 1;

    ctx.beginPath();

    ctx.moveTo(
        leftMargin,
        beatY
    );

    ctx.lineTo(
        canvas.width - rightMargin,
        beatY
    );

    ctx.stroke();


    // ========================================================
    // SHORT PULSING BEATS
    // ========================================================

    beats.forEach((beat, index) => {{

        const x = timeToX(beat);

        const distance = currentTime - beat;


        // -----------------------------------------------
        // 0.10 SECOND PULSE
        // -----------------------------------------------

        const active =
            distance >= 0 &&
            distance < 0.10;


        let radius = 6;


        if (active) {{

            const progress =
                distance / 0.10;

            radius =
                13 - (progress * 7);
        }}


        // -----------------------------------------------
        // Beat circle
        // -----------------------------------------------

        ctx.beginPath();

        ctx.arc(
            x,
            beatY,
            radius,
            0,
            Math.PI * 2
        );

        ctx.fillStyle =
            active
            ? "#c94f7c"
            : "#777";

        ctx.fill();


        // -----------------------------------------------
        // Beat number
        // -----------------------------------------------

        ctx.fillStyle =
            active
            ? "#c94f7c"
            : "#555";

        ctx.font =
            active
            ? "bold 12px Arial"
            : "10px Arial";

        ctx.fillText(
            index + 1,
            x - 3,
            beatY + 22
        );

    }});


    // ========================================================
    // PLAYHEAD
    // ========================================================

    const playheadX =
        timeToX(currentTime);

    ctx.strokeStyle = "#222";
    ctx.lineWidth = 2;

    ctx.beginPath();

    ctx.moveTo(
        playheadX,
        topMargin
    );

    ctx.lineTo(
        playheadX,
        beatY + 15
    );

    ctx.stroke();


    // ========================================================
    // CURRENT TIME
    // ========================================================

    ctx.fillStyle = "#222";
    ctx.font = "14px Arial";

    ctx.fillText(
        currentTime.toFixed(2) +
        " / " +
        totalTime.toFixed(2) +
        " sec",
        canvas.width - 140,
        25
    );


    // ========================================================
    // CONTINUE
    // ========================================================

    requestAnimationFrame(draw);

}}


// ============================================================
// START
// ============================================================

draw();

</script>
"""

display(HTML(html))